In [ ]:
# Set this to the public HTTPS URL of your product image, then click Run All once.
IMAGE_URL = "PASTE_PUBLIC_PRODUCT_IMAGE_URL_HERE"

import os, torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video

if not IMAGE_URL.startswith("https://") or "PASTE_" in IMAGE_URL:
    raise ValueError("Set IMAGE_URL to your public HTTPS product image URL first.")
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is not enabled. Select GPU in Notebook Settings and run again.")

MODEL_ID = "stabilityai/stable-video-diffusion-img2vid-xt"
CACHE_DIR = "/root/.cache/huggingface"
os.makedirs("/kaggle/working", exist_ok=True)
print("Loading SVD-XT in FP16 with CPU offload; first run downloads model weights.")
pipe = StableVideoDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16",
    cache_dir=CACHE_DIR,
)
pipe.enable_model_cpu_offload()
pipe.unet.enable_forward_chunking()

image = load_image(IMAGE_URL).convert("RGB").resize((1024, 576))
generator = torch.Generator(device="cuda").manual_seed(42)
print("Generating 25 frames. This can take a while on Tesla T4.")
frames = pipe(
    image,
    num_frames=25,
    decode_chunk_size=2,
    motion_bucket_id=127,
    noise_aug_strength=0.02,
    generator=generator,
).frames[0]

output_path = "/kaggle/working/click_collect_ai_video.mp4"
export_to_video(frames, output_path, fps=7)
print("VIDEO READY:", output_path)
print("Download click_collect_ai_video.mp4 from the Kaggle Output panel.")
